# Run a review with `run_review`

The smallest way to drive the whole pipeline from Python: build a `ReviewRequest`, call
`agentic_review.agents.pipeline_agent.run_review`, and look at what comes back. This is the same call the
`agentic-review review` command makes.

- Select this project's `.venv` kernel (`uv sync` installs `ipykernel`).
- Put the provider key in `.env` as `AGENTIC_REVIEW_API_KEY` (`GROQ_API_KEY` is used as a fallback).
- **Langfuse** (optional): `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` and `LANGFUSE_BASE_URL` in `.env`. With
  them, every LangGraph/LangChain run is traced and the system prompt comes from Langfuse.
- `REPO` must be a git repository with uncommitted changes. The default is the demo built by
  `scripts/make_review_demo.py`.

## 1 · Settings

In [ ]:
import os
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# ---- change these -------------------------------------------------------------------------------------------------
REPO = PROJECT.parent / "review-demo"  # the repository to review
SCOPE = None                           # None = config/default, or "working-tree" | "staged" | "range"
BASE = None                            # the base ref, only for SCOPE = "range"
MODEL = None                           # None = use the repository's .agentic-review.yml (or the built-in default)
BASE_URL = None                        # e.g. "https://api.groq.com/openai/v1"
# -------------------------------------------------------------------------------------------------------------------


def read_env_file(path: Path) -> dict[str, str]:
    values = {}
    if path.is_file():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, _, value = line.partition("=")
                values[key.strip()] = value.strip().strip('"')
    return values


ENV = {**os.environ, **read_env_file(PROJECT / ".env")}
if not ENV.get("AGENTIC_REVIEW_API_KEY") and ENV.get("GROQ_API_KEY"):
    ENV["AGENTIC_REVIEW_API_KEY"] = ENV["GROQ_API_KEY"]

# Langfuse is read from os.environ, not from ENV: by its client and by the system-prompt loader.
for name in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"):
    if ENV.get(name):
        os.environ.setdefault(name, ENV[name])

print("repo:   ", REPO, "(exists)" if REPO.is_dir() else "(MISSING)")
print("api key:", "set" if ENV.get("AGENTIC_REVIEW_API_KEY") else "NOT SET")  # the value is never printed

## 1b · Langfuse tracing

A Langfuse `CallbackHandler` is attached to **every** LangChain/LangGraph run in this kernel through a configure hook,
so the pipeline graph, the review graph and each model call are traced without changing any code. Graph state carries
no credential (`tests/integration/test_trace_carries_no_credential.py`), so nothing secret is shipped.

Without the Langfuse keys this cell does nothing, and the built-in system prompt is used.

In [ ]:
from contextvars import ContextVar

from langchain_core.tracers.context import register_configure_hook

from agentic_review.core.assembly.prompt import load_instructions

LANGFUSE_ON = bool(os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY"))
langfuse = tracer = None
if LANGFUSE_ON:
    from langfuse import get_client
    from langfuse.langchain import CallbackHandler

    langfuse = get_client()
    try:
        auth_ok = langfuse.auth_check()
    except Exception as error:  # the server is down (e.g. the local Docker stack is not running)
        auth_ok = False
        print("Langfuse unreachable:", type(error).__name__, error)
    print("Langfuse auth:", "ok" if auth_ok else "FAILED", "|", os.environ.get("LANGFUSE_BASE_URL"))
    if not auth_ok:
        langfuse = None
if langfuse is not None:
    if "_tracer_var" not in globals():  # register once: re-running this cell must not trace every run twice
        _tracer_var: ContextVar = ContextVar("notebook_langfuse_tracer", default=None)
        register_configure_hook(_tracer_var, inheritable=True)
    tracer = CallbackHandler()
    _tracer_var.set(tracer)  # from here on, every LangChain/LangGraph run is traced
else:
    if "_tracer_var" in globals():
        _tracer_var.set(None)  # a re-run with Langfuse gone stops tracing
    print("no tracing" if LANGFUSE_ON else "Langfuse not configured: no tracing")

loaded = load_instructions(os.environ)
print(f"system prompt: {loaded.source}" + (f"  -- fell back: {loaded.problem}" if loaded.problem else ""))

## 2 · Run the review

`port_factory` is optional. It is used here only so `MODEL` / `BASE_URL` above can override the repository's
configuration without editing any file; leave both `None` and `run_review` uses its default.

In [ ]:
import time

from agentic_review.agents.pipeline_agent import ReviewRequest, Reviewed, run_review
from agentic_review.agents.review_port import build_review_port


def override_model(agent_settings, credential):
    overrides = {k: v for k, v in {"model": MODEL, "base_url": BASE_URL}.items() if v}
    return build_review_port(agent_settings.model_copy(update=overrides), credential)


request = ReviewRequest(cwd=REPO, scope=SCOPE, base=BASE)

started = time.perf_counter()
result = run_review(request, env=ENV, port_factory=override_model if (MODEL or BASE_URL) else None)
print(f"{type(result).__name__} in {time.perf_counter() - started:.1f}s")

if langfuse is not None:
    langfuse.flush()  # send the trace now rather than at interpreter exit
    print("trace:", langfuse.get_trace_url(trace_id=tracer.last_trace_id) if tracer.last_trace_id else "none recorded")

## 3 · Read the result

`run_review` returns one of two things:

- `PreflightFailure` — the repository, configuration or credential was not usable; nothing was reviewed.
- `Reviewed` — a `ReviewReport`, even when something degraded (FR-11.2). Always read `degradations`.

In [ ]:
from rich.console import Console

from agentic_review.core.output.terminal import render_terminal

if not isinstance(result, Reviewed):
    print("PREFLIGHT FAILED:\n" + result.message)
else:
    report = result.report
    render_terminal(report, Console(width=140, force_jupyter=False, force_terminal=False, no_color=True))
    if result.ignored_settings:
        print("ignored settings:", ", ".join(result.ignored_settings))

In [ ]:
assert isinstance(result, Reviewed), "preflight failed - see the cell above"
report = result.report

print("verdict:   ", report.verdict)
print("completion:", report.run.completion_status)
print("model:     ", report.run.model, "@", report.run.provider)
print("tokens:    ", report.run.tokens_in, "in /", report.run.tokens_out, "out")
print("stages ms: ", report.run.stage_durations_ms.model_dump(by_alias=True, exclude_none=True))

print(f"\n{len(report.findings)} finding(s)")
for f in report.findings:
    print(f"- [{f.severity}/{f.category}] {f.file}:{f.start_line}-{f.end_line}  {f.rationale}")

print(f"\n{len(report.degradations)} degradation(s)" + ("  (a clean run)" if not report.degradations else ""))
for d in report.degradations:
    print(f"- {d.source} {d.code}: {d.detail}")

## 4 · The full report as JSON

The same document `agentic-review review --json` prints.

In [ ]:
from agentic_review.core.output.json_out import to_json

print(to_json(report))